In [1]:
pip install tensorflow rasterio scikit-learn matplotlib pandas numpy

Note: you may need to restart the kernel to use updated packages.


# TimeFormer

In [2]:
import numpy as np
import pandas as pd
import rasterio
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split as sk_split
import gc
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter
import math
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
BASE_DIR = '/kaggle/input/datasets/fedyou/smartsdg/SmartSDG-Tunisia/Brut_Data'
CDI_PATH = '/kaggle/input/datasets/fedyou/smartsdg/SmartSDG-Tunisia/CDI'

NUM_TILES = 90

# ── Hyperparamètres allégés ──────────────────────────────────
SEQUENCE_LENGTH = 3
D_MODEL         = 32  
N_HEADS         = 2  
NUM_LAYERS      = 1    
D_FF            = 64    
DROPOUT         = 0.3 
EPOCHS          = 20 
BATCH_SIZE      = 64   
LEARNING_RATE   = 0.001
PATIENCE        = 5 
RANDOM_SEED     = 42
PATCH_SIZE      = 8
TRAIN_RATIO = 0.64
VAL_RATIO   = 0.30
DURATION_MONTHS = 311
NUM_SCALES = 2     
GAMMA      = 0.1    

torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)


# ============================================================
# DATE UTILITIES
# ============================================================
def get_last_n_months(n_months=60):
    end_date   = datetime(2025, 12, 1)
    start_date = end_date - relativedelta(months=n_months - 1)
    dates, current = [], start_date
    while current <= end_date:
        dates.append(current)
        current += relativedelta(months=1)
    return sorted(dates)


# ============================================================
# FILE LOADING
# ============================================================
def _fmt(date):
    return date.strftime('%Y'), date.strftime('%m'), date.strftime('%Y_%m')

REF_SHAPE = REF_TRANSFORM = REF_CRS = None

def _resample_to_ref(src, ref_shape, ref_transform, ref_crs):
    from rasterio.warp import reproject, Resampling
    data_out = np.empty(ref_shape, dtype=np.float32)
    reproject(
        source=rasterio.band(src, 1), destination=data_out,
        src_transform=src.transform, src_crs=src.crs,
        dst_transform=ref_transform, dst_crs=ref_crs,
        resampling=Resampling.bilinear
    )
    return data_out

def load_cdi_raster(date):
    _, _, ym = _fmt(date)
    path = os.path.join(CDI_PATH, f'CDI_Tunisia_{ym}.tif')
    if not os.path.exists(path):
        return None, None
    try:
        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)
            if REF_SHAPE is not None and arr.shape != REF_SHAPE:
                arr = _resample_to_ref(src, REF_SHAPE, REF_TRANSFORM, REF_CRS)
            return arr, src.transform
    except Exception:
        return None, None

def load_monthly_indicators(date):
    global REF_SHAPE, REF_TRANSFORM, REF_CRS
    _, _, ym = _fmt(date)
    paths = {
        'SPI':  os.path.join(BASE_DIR, 'SPI',  f'Tunisia_SPI_{ym}.tif'),
        'SPEI': os.path.join(BASE_DIR, 'SPEI', f'Tunisia_SPEI_{ym}.tif'),
        'SM':   os.path.join(BASE_DIR, 'SM',   f'GLDAS_SoilMoisture_Tunisia_{ym}.tif'),
        'NDVI': os.path.join(BASE_DIR, 'NDVI', f'NDVI_Tunisia_{ym}.tif'),
        'LST':  os.path.join(BASE_DIR, 'LST',  f'LST_MonthlyMean_{ym}.tif'),
    }
    indicators = {}
    for name, path in paths.items():
        if not os.path.exists(path):
            return None, None, False
        try:
            with rasterio.open(path) as src:
                if REF_SHAPE is None:
                    REF_SHAPE     = (src.height, src.width)
                    REF_TRANSFORM = src.transform
                    REF_CRS       = src.crs
                arr = src.read(1).astype(np.float32)
                if arr.shape != REF_SHAPE:
                    arr = _resample_to_ref(src, REF_SHAPE, REF_TRANSFORM, REF_CRS)
                indicators[name] = arr
        except Exception:
            return None, None, False
    return indicators, REF_TRANSFORM, True


# ============================================================
# TEMPORAL DATA COLLECTION
# ============================================================
def collect_temporal_data(n_months=DURATION_MONTHS):
    print(f"\n📅 Collecting data for the last {n_months} months...")
    all_dates = get_last_n_months(n_months)
    print(f"   Period: {all_dates[0]:%Y-%m} → {all_dates[-1]:%Y-%m}")
    temporal_data, raster_transform = [], None
    for date in tqdm(all_dates, desc="Loading"):
        indicators, transform, ok = load_monthly_indicators(date)
        if not ok:
            continue
        cdi_real, _ = load_cdi_raster(date)
        if cdi_real is None:
            continue
        temporal_data.append({'date': date, 'indicators': indicators, 'cdi_real': cdi_real})
        if raster_transform is None:
            raster_transform = transform
    print(f"   ✅ {len(temporal_data)} months available\n")
    return temporal_data, raster_transform


def generate_tiles(raster_shape, num_tiles=NUM_TILES):
    H, W      = raster_shape
    tile_size = max(1, H // num_tiles)
    tiles     = []
    for r in range(0, H, tile_size):
        r_end = min(r + tile_size, H)
        tiles.append((r, r_end, 0, W))
    if len(tiles) > num_tiles:
        last_r0 = tiles[num_tiles - 1][0]
        tiles   = tiles[:num_tiles - 1] + [(last_r0, H, 0, W)]
    return tiles


INDICATOR_KEYS = ['SPI', 'SPEI', 'SM', 'NDVI', 'LST']
NUM_FEATURES   = len(INDICATOR_KEYS)


# ============================================================
# SPATIO-TEMPORAL SEQUENCES
# ============================================================
def prepare_sequences_tile_timeformer(temporal_data, r0, r1, c0, c1, patch_size=PATCH_SIZE):
    temporal_data = sorted(temporal_data, key=lambda x: x['date'])
    num_months    = len(temporal_data)
    H_tile, W_tile = r1 - r0, c1 - c0
    sequences, targets, seq_dates, patch_origins = [], [], [], []

    for target_idx in range(SEQUENCE_LENGTH, num_months):
        target_month    = temporal_data[target_idx]
        sequence_months = temporal_data[target_idx - SEQUENCE_LENGTH:target_idx]

        seq_stack    = np.zeros((SEQUENCE_LENGTH, NUM_FEATURES, H_tile, W_tile), dtype=np.float32)
        valid_global = True
        for t, md in enumerate(sequence_months):
            for c_idx, k in enumerate(INDICATOR_KEYS):
                pd_ = md['indicators'][k][r0:r1, c0:c1]
                if np.any(np.isnan(pd_)) or np.any(~np.isfinite(pd_)) or np.any(np.abs(pd_) > 1e6):
                    valid_global = False; break
                seq_stack[t, c_idx] = pd_
            if not valid_global:
                break
        if not valid_global:
            continue

        tgt_patch = target_month['cdi_real'][r0:r1, c0:c1].astype(np.int64)
        if np.any(np.isnan(target_month['cdi_real'][r0:r1, c0:c1])):
            continue
        tgt_patch = np.clip(tgt_patch, 0, 5)

        for pr in range(0, H_tile, patch_size):
            for pc in range(0, W_tile, patch_size):
                pr_end = min(pr + patch_size, H_tile)
                pc_end = min(pc + patch_size, W_tile)
                ph, pw = pr_end - pr, pc_end - pc
                sub_seq = np.zeros((SEQUENCE_LENGTH, NUM_FEATURES, patch_size, patch_size), dtype=np.float32)
                sub_tgt = np.zeros((patch_size, patch_size), dtype=np.int64)
                sub_seq[:, :, :ph, :pw] = seq_stack[:, :, pr:pr_end, pc:pc_end]
                sub_tgt[:ph, :pw]       = tgt_patch[pr:pr_end, pc:pc_end]
                sequences.append(sub_seq)
                targets.append(sub_tgt)
                seq_dates.append(target_month['date'])
                patch_origins.append((r0 + pr, c0 + pc, ph, pw))

    if len(sequences) == 0:
        return None, None, None, None
    return (np.array(sequences, dtype=np.float32),
            np.array(targets,   dtype=np.int64),
            np.array(seq_dates),
            np.array(patch_origins))

def split_data_ordered(sequences, targets, seq_dates, patch_origins):
    unique_dates = np.array(sorted(set(seq_dates)))
    n_dates      = len(unique_dates)
    n_test_dates = max(1, int(n_dates * (1 - TRAIN_RATIO - VAL_RATIO)))
    split_date   = unique_dates[-n_test_dates]

    test_mask = seq_dates >= split_date
    tv_mask   = seq_dates <  split_date

    X_tv, y_tv = sequences[tv_mask],   targets[tv_mask]
    X_te, y_te = sequences[test_mask], targets[test_mask]
    origins_te = patch_origins[test_mask]
    dates_te   = seq_dates[test_mask]

    if len(X_tv) == 0 or len(X_te) == 0:
        return None

    val_ratio_in_tv = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    tr_idx, va_idx  = sk_split(np.arange(len(X_tv)), test_size=val_ratio_in_tv,
                                random_state=RANDOM_SEED, shuffle=True)
    return (X_tv[tr_idx], X_tv[va_idx], X_te,
            y_tv[tr_idx], y_tv[va_idx], y_te,
            origins_te, dates_te)

def normalize_features_spatial(X_train, X_val, X_test):
    N, T, C, pH, pW = X_train.shape
    X_tr_s = X_train.copy()
    X_va_s = X_val.copy()
    X_te_s = X_test.copy()
    for c_idx in range(C):
        flat = X_train[:, :, c_idx, :, :].reshape(-1, 1)
        sc   = RobustScaler().fit(flat)
        X_tr_s[:, :, c_idx, :, :] = sc.transform(flat).reshape(N,  T, pH, pW)
        X_va_s[:, :, c_idx, :, :] = sc.transform(X_val [:, :, c_idx, :, :].reshape(-1,1)).reshape(len(X_val),  T, pH, pW)
        X_te_s[:, :, c_idx, :, :] = sc.transform(X_test[:, :, c_idx, :, :].reshape(-1,1)).reshape(len(X_test), T, pH, pW)
    return X_tr_s, X_va_s, X_te_s


class CDIDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self):          return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


# ============================================================
#  ██████╗  █████╗  ██████╗  ███████╗ ██████╗
# ██╔════╝ ██╔══██╗ ██╔══██╗ ██╔════╝ ██╔══██╗
# ██║      ███████║ ██████╔╝ ███████╗ ██████╔╝
# ██║      ██╔══██║ ██╔═══╝  ╚════██║ ██╔══██╗
# ╚██████╗ ██║  ██║ ██║      ███████║ ██║  ██║
#  ╚═════╝ ╚═╝  ╚═╝ ╚═╝      ╚══════╝ ╚═╝  ╚═╝
#
# Architecture fidèle au papier — version allégée & rapide
# MoSA  = Hawkes decay (γ) + Causal mask  (§3.4)
# Block = 1D-CNN embed → Seg (Ps=Ks=⌈√Ls⌉) → Intra-MoSA → Inter-MoSA
# Multi-scale : S=2 échelles via AvgPool  (§3.2)
# ============================================================

_hawkes_cache: dict = {}

def hawkes_matrix(T: int, gamma: float, device: torch.device) -> torch.Tensor:
    key = (T, gamma, str(device))
    if key not in _hawkes_cache:
        idx   = torch.arange(T, dtype=torch.float32, device=device)
        diff  = idx.unsqueeze(1) - idx.unsqueeze(0)   # diff[i,j] = i - j
        omega = torch.exp(-gamma * diff.clamp(min=0))  # decay quand j < i
        _hawkes_cache[key] = omega
    return _hawkes_cache[key]


# ── MoSA : Self-Attention with Modulation Term  (§3.4) ───────────────
class MoSA(nn.Module):
    """
    Scaled dot-product attention + Hawkes decay + causal mask.
    Léger : une seule tête, pas de projection O séparée (fusionnée dans W_o).
    """
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1, gamma: float = GAMMA):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.gamma   = gamma
        self.scale   = self.d_head ** -0.5

        # Fused QKV projection — 1 seul matmul au lieu de 3
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)   # LayerNorm plus stable que BatchNorm ici

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, T, D)"""
        B, T, D = x.shape
        residual = x

        # ── QKV fused ────────────────────────────────────────────────────
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)          # (3, B, H, T, d)
        Q, K, V = qkv[0], qkv[1], qkv[2]           # each (B, H, T, d)

        # ── Scaled dot-product  [Eq.11] ───────────────────────────────────
        A = torch.matmul(Q, K.transpose(-2, -1)) * self.scale   # (B, H, T, T)

        # ── Hawkes decay  [Eq.12-13] ─────────────────────────────────────
        omega = hawkes_matrix(T, self.gamma, x.device)           # (T, T)
        A = A * omega.unsqueeze(0).unsqueeze(0)

        # ── Causal mask  [Eq.14] ─────────────────────────────────────────
        mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
        A    = A.masked_fill(~mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        # ── Softmax ───────────────────────────────────────────────────────
        A = torch.softmax(A, dim=-1)
        A = torch.nan_to_num(A, nan=0.0)
        A = self.drop(A)

        # ── Output ────────────────────────────────────────────────────────
        out = torch.matmul(A, V)                                   # (B, H, T, d)
        out = out.transpose(1, 2).contiguous().view(B, T, D)       # (B, T, D)
        out = self.W_o(out)
        return self.norm(out + residual)


# ── FFN léger avec résidu + LayerNorm ────────────────────────────────
class FFN(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net  = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        return self.norm(x + self.net(x))


# ── TimeFormer pour UNE échelle  (§3.3) ──────────────────────────────
class TimeFormerScale(nn.Module):
    """
    Traite une série X^s de longueur Ls :
      1. Embedding 1D-CNN             §3.3.1
      2. Segmentation Ps=Ks=⌈√Ls⌉    §3.3.2
      3. Intra-patch MoSA + FFN       §3.3.3
      4. Inter-patch MoSA + FFN → Z^s §3.3.4
    Les projections intra/inter sont construites une seule fois
    avec la taille réelle (pas de hasattr dynamique → plus rapide).
    """
    def __init__(self, input_len: int, d_model: int, n_heads: int,
                 d_ff: int, n_layers: int, dropout: float, gamma: float):
        super().__init__()
        self.d_model = d_model

        # ── §3.3.1  Embedding 1D-CNN ──────────────────────────────────────
        self.embed = nn.Sequential(
            nn.Conv1d(1, d_model, kernel_size=3, padding=1),
            nn.GELU(),
        )

        # ── §3.3.2  Calcul Ps, Ks ────────────────────────────────────────
        Ks = math.ceil(math.sqrt(input_len))
        Ps = Ks
        self.Ks = Ks
        self.Ps = Ps
        self.padded_len = Ps * Ks

        # ── §3.3.3  Intra-patch MoSA (time-step as token) ─────────────────
        self.intra_mosa = nn.ModuleList([MoSA(d_model, n_heads, dropout, gamma)
                                         for _ in range(n_layers)])
        self.intra_ffn  = FFN(d_model, d_ff, dropout)
        # FNN flatten Ks*D → D  [Eq.7]
        self.intra_proj = nn.Linear(Ks * d_model, d_model, bias=False)

        # ── §3.3.4  Inter-patch MoSA (patch as token) ─────────────────────
        self.inter_mosa = nn.ModuleList([MoSA(d_model, n_heads, dropout, gamma)
                                         for _ in range(n_layers)])
        self.inter_ffn  = FFN(d_model, d_ff, dropout)
        # FNN flatten Ps*D → D  [Eq.9]
        self.inter_proj = nn.Linear(Ps * d_model, d_model, bias=False)

    def forward(self, x_s: torch.Tensor) -> torch.Tensor:
        """x_s : (B, Ls)  →  Z^s : (B, D)"""
        B, Ls = x_s.shape

        # ── Embedding ─────────────────────────────────────────────────────
        emb = self.embed(x_s.unsqueeze(1))          # (B, D, Ls)
        emb = emb.permute(0, 2, 1)                  # (B, Ls, D)

        # ── Zero-padding → Ps*Ks ─────────────────────────────────────────
        if self.padded_len > Ls:
            pad = torch.zeros(B, self.padded_len - Ls, self.d_model, device=x_s.device)
            emb = torch.cat([emb, pad], dim=1)
        else:
            emb = emb[:, :self.padded_len, :]

        # ── §3.3.3  Intra-patch MoSA ──────────────────────────────────────
        patches = emb.view(B * self.Ps, self.Ks, self.d_model)  # (B*Ps, Ks, D)
        for layer in self.intra_mosa:
            patches = layer(patches)
        patches = self.intra_ffn(patches)
        # flatten Ks×D → D
        tokens = self.intra_proj(patches.reshape(B * self.Ps, self.Ks * self.d_model))
        tokens = tokens.view(B, self.Ps, self.d_model)           # (B, Ps, D)

        # ── §3.3.4  Inter-patch MoSA ──────────────────────────────────────
        for layer in self.inter_mosa:
            tokens = layer(tokens)
        tokens = self.inter_ffn(tokens)
        # flatten Ps×D → D
        Z = self.inter_proj(tokens.reshape(B, self.Ps * self.d_model))  # (B, D)
        return Z


# ── CDI TimeFormer Classifier  (§3.2 — Multi-scale) ──────────────────
class CDITimeFormerClassifier(nn.Module):
    """
    Input  : (B, T, C, H, W)
    Sortie : (B, num_classes, H, W)

    • Pixel series = T*C valeurs  (série univariée par pixel)
    • S échelles via AvgPool1d
    • Concat Z^s → classifieur léger
    """
    def __init__(self,
                 seq_len=SEQUENCE_LENGTH, num_vars=NUM_FEATURES,
                 d_model=D_MODEL, n_heads=N_HEADS, num_layers=NUM_LAYERS,
                 d_ff=D_FF, num_classes=6, dropout=DROPOUT,
                 num_scales=NUM_SCALES, gamma=GAMMA):
        super().__init__()
        self.num_classes = num_classes
        self.num_scales  = num_scales

        base_len = seq_len * num_vars   # longueur série à l'échelle 1

        # Un TimeFormerScale par échelle — tailles pré-calculées
        self.scale_blocks = nn.ModuleList()
        for s in range(1, num_scales + 1):
            ls = math.ceil(base_len / s)
            self.scale_blocks.append(
                TimeFormerScale(ls, d_model, n_heads, d_ff, num_layers, dropout, gamma)
            )

        # Projection finale
        self.classifier = nn.Sequential(
            nn.Linear(num_scales * d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, T, C, H, W)"""
        B, T, C, H, W = x.shape

        # ── Série par pixel ────────────────────────────────
        x_pix = x.permute(0, 3, 4, 1, 2).reshape(B * H * W, T * C)

        # ── Multi-scale  [Eq.3] ───────────────────────────────────────────
        Z_list = []
        for s_idx, block in enumerate(self.scale_blocks):
            kernel = s_idx + 1
            if kernel == 1:
                x_s = x_pix
            else:
                x_s = torch.nn.functional.avg_pool1d(
                    x_pix.unsqueeze(1), kernel_size=kernel, stride=kernel
                ).squeeze(1)
            Z_list.append(block(x_s))              

        # ── Classifier ────────────────────────────────────────────────────
        Z_cat  = torch.cat(Z_list, dim=-1)        
        logits = self.classifier(Z_cat)              
        logits = logits.view(B, H, W, self.num_classes).permute(0, 3, 1, 2)
        return logits                               


def train_model(train_loader, val_loader, model, criterion, optimizer):
    best_val_acc, patience_counter = 0.0, 0
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    for epoch in range(EPOCHS):
        model.train()
        for bX, by in train_loader:
            bX, by = bX.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(bX)
                loss   = criterion(logits, by)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for bX, by in val_loader:
                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    logits = model(bX.to(DEVICE))
                pred = logits.argmax(dim=1)
                val_preds.extend(pred.cpu().numpy().flatten())
                val_true.extend(by.numpy().flatten())

        val_acc = accuracy_score(val_true, val_preds)
        if val_acc > best_val_acc:
            best_val_acc, patience_counter = val_acc, 0
            torch.save(model.state_dict(), 'best_tile.pth')
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

        if epoch % 5 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return best_val_acc

CLASS_NAMES = ['Normal', 'Watch', 'Warning', 'Alert-1', 'Alert-2', 'Urgency']

def process_tile(temporal_data, tile_idx, r0, r1, c0, c1):
    seqs, tgts, dates, origins = prepare_sequences_tile_timeformer(
        temporal_data, r0, r1, c0, c1, patch_size=PATCH_SIZE)

    if seqs is None or len(seqs) < 10:
        return None

    split = split_data_ordered(seqs, tgts, dates, origins)
    if split is None:
        return None

    X_tr, X_va, X_te, y_tr, y_va, y_te, origins_te, dates_te = split
    del seqs; gc.collect()

    X_tr_s, X_va_s, X_te_s = normalize_features_spatial(X_tr, X_va, X_te)
    del X_tr, X_va, X_te; gc.collect()

    loader_kw = dict(batch_size=BATCH_SIZE, shuffle=False,
                     pin_memory=True, num_workers=2)
    tr_loader = DataLoader(CDIDataset(X_tr_s, y_tr), **loader_kw)
    va_loader = DataLoader(CDIDataset(X_va_s, y_va), **loader_kw)
    te_loader = DataLoader(CDIDataset(X_te_s, y_te), **loader_kw)
    del X_tr_s, X_va_s, X_te_s; gc.collect()

    model = CDITimeFormerClassifier(
        seq_len=SEQUENCE_LENGTH, num_vars=NUM_FEATURES,
        d_model=D_MODEL, n_heads=N_HEADS, num_layers=NUM_LAYERS,
        d_ff=D_FF, num_classes=6, dropout=DROPOUT,
        num_scales=NUM_SCALES, gamma=GAMMA,
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_val_acc = train_model(tr_loader, va_loader, model, criterion, optimizer)

    model.load_state_dict(torch.load('best_tile.pth'))
    model.eval()

    preds_flat, trues_flat = [], []
    with torch.no_grad():
        for bX, by in te_loader:
            logits = model(bX.to(DEVICE))
            pred   = logits.argmax(dim=1)
            preds_flat.extend(pred.cpu().numpy().flatten())
            trues_flat.extend(by.numpy().flatten())

    preds_flat = np.array(preds_flat)
    trues_flat = np.array(trues_flat)

    acc = accuracy_score(trues_flat, preds_flat)
    kap = cohen_kappa_score(trues_flat, preds_flat)
    f1m = f1_score(trues_flat, preds_flat, average='macro',    zero_division=0)
    f1w = f1_score(trues_flat, preds_flat, average='weighted', zero_division=0)

    # ── Reconstruction des cartes spatiales ───────────────────────────
    pred_maps_by_date, true_maps_by_date = {}, {}
    H_tile, W_tile = r1 - r0, c1 - c0
    patch_counter  = 0

    model.eval()
    with torch.no_grad():
        for bX, by in te_loader:
            logits     = model(bX.to(DEVICE))
            batch_pred = logits.argmax(dim=1).cpu().numpy()
            batch_true = by.numpy()
            for b in range(batch_pred.shape[0]):
                global_idx = patch_counter + b
                if global_idx >= len(origins_te):
                    break
                r_orig, c_orig, ph, pw = origins_te[global_idx]
                date_key               = dates_te[global_idx]
                if date_key not in pred_maps_by_date:
                    pred_maps_by_date[date_key] = np.full((H_tile, W_tile), np.nan, dtype=np.float32)
                    true_maps_by_date[date_key] = np.full((H_tile, W_tile), np.nan, dtype=np.float32)
                pr_local = r_orig - r0
                pc_local = c_orig - c0
                pred_maps_by_date[date_key][pr_local:pr_local+ph, pc_local:pc_local+pw] = batch_pred[b][:ph, :pw]
                true_maps_by_date[date_key][pr_local:pr_local+ph, pc_local:pc_local+pw] = batch_true[b][:ph, :pw]
            patch_counter += batch_pred.shape[0]

    del model, tr_loader, va_loader, te_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        'tile_idx': tile_idx, 'bounds': (r0, r1, c0, c1),
        'n_test': len(trues_flat),
        'accuracy': acc, 'kappa': kap,
        'f1_macro': f1m, 'f1_weighted': f1w,
        'val_acc': best_val_acc,
        'pred_maps_by_date': pred_maps_by_date,
        'true_maps_by_date': true_maps_by_date,
    }


def print_global_summary(results, full_shape, output_dir='.'):
    if not results:
        print("No results available."); return
    results_with_map = [r for r in results if r.get('pred_maps_by_date')]
    df = pd.DataFrame([{k: v for k, v in r.items()
                        if k not in ('pred_maps_by_date','true_maps_by_date')}
                       for r in results])
    total_samples = df['n_test'].sum()
    w    = df['n_test'].values / total_samples
    wacc = np.average(df['accuracy'].values,    weights=w)
    wkap = np.average(df['kappa'].values,       weights=w)
    wf1m = np.average(df['f1_macro'].values,    weights=w)
    wf1w = np.average(df['f1_weighted'].values, weights=w)

    print("\n" + "="*50)
    print(f"  Metric            Simple")
    print(f"  {'-'*30}")
    print(f"  Accuracy          {df['accuracy'].mean():.4f}")
    print(f"  Cohen Kappa       {df['kappa'].mean():.4f}")
    print(f"  F1 Macro          {df['f1_macro'].mean():.4f}")
    print(f"  F1 Weighted       {df['f1_weighted'].mean():.4f}")
    print("="*50)
# ============================================================
# EXÉCUTION
# ============================================================
OUTPUT_DIR = '/kaggle/working/cdi_rasters_timeformer'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n" + "="*70)
print("🚀 CDI PREDICTION — TimeFormer FAST (Paper Architecture Allégée)")
print(f"   d_model        : {D_MODEL}    |  Heads : {N_HEADS}   |  Layers : {NUM_LAYERS}")
print(f"   d_ff           : {D_FF}   |  Scales (S) : {NUM_SCALES}  |  γ : {GAMMA}")
print(f"   Batch          : {BATCH_SIZE}   |  Epochs : {EPOCHS}  |  Patience : {PATIENCE}")
print(f"   AMP (fp16)     : {torch.cuda.is_available()}")
print(f"   MoSA           : Hawkes decay (γ={GAMMA}) + Causal mask")
print(f"   Segmentation   : Ps = Ks = ⌈√Ls⌉  (paper §3.3.2)")
print(f"   CDI classes    : {', '.join(CLASS_NAMES)}")
print("="*70)

temporal_data, raster_transform = collect_temporal_data(DURATION_MONTHS)
full_shape = list(temporal_data[0]['indicators'].values())[0].shape
print(f"\n   Raster shape: {full_shape}")

tiles = generate_tiles(full_shape, NUM_TILES)
print("\n🔄 Model Training...\n")

all_results = []
for tile_idx, (r0, r1, c0, c1) in enumerate(tiles):
    result = process_tile(temporal_data, tile_idx, r0, r1, c0, c1)
    if result is not None:
        all_results.append(result)

print_global_summary(all_results, full_shape, output_dir=OUTPUT_DIR)
print("\n✅ DONE")

Device: cuda

🚀 CDI PREDICTION — TimeFormer FAST (Paper Architecture Allégée)
   d_model        : 32    |  Heads : 2   |  Layers : 1
   d_ff           : 64   |  Scales (S) : 2  |  γ : 0.1
   Batch          : 64   |  Epochs : 20  |  Patience : 5
   AMP (fp16)     : True
   MoSA           : Hawkes decay (γ=0.1) + Causal mask
   Segmentation   : Ps = Ks = ⌈√Ls⌉  (paper §3.3.2)
   CDI classes    : Normal, Watch, Warning, Alert-1, Alert-2, Urgency

📅 Collecting data for the last 311 months...
   Period: 2000-02 → 2025-12


Loading: 100%|██████████| 311/311 [01:07<00:00,  4.64it/s]


   ✅ 311 months available


   Raster shape: (818, 455)

🔄 Model Training...


  Metric            Simple
  ------------------------------
  Accuracy          0.9063
  Cohen Kappa       0.7481
  F1 Macro          0.5032
  F1 Weighted       0.9006

✅ DONE
